# 07 — Evaluate All Models on Adverse Conditions

Evaluates all 4 models on six adverse test splits. Each split isolates
a single degradation variable:

| Split | Degradation | ~Images |
|---|---|---|
| `rainy_day` | weather (rain), daytime only | 396 |
| `snowy_day` | weather (snow), daytime only | 422 |
| `night_clear` | lighting (night), clear weather only | 3 274 |
| `overcast_day` | weather (overcast), daytime only | 1 039 |
| `partly_cloudy_day` | weather (partly cloudy), daytime only | 638 |
| `dawn_dusk_clear` | lighting (dawn/dusk), clear weather only | 307 |

Robustness metrics (mAP drop, retention %) are computed relative to the
clear-day baseline loaded from `results/clear_day/scores.json`.

**Prerequisites:** notebook 06 must have completed successfully.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

import json
import torch
from pathlib import Path
from src.eval_utils import compute_map, compute_precision_recall, compute_robustness_metrics
from src.train_utils import setup_logging

logger = setup_logging()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

clear_scores = json.loads(Path('../results/clear_day/scores.json').read_text())
print('Loaded clear-day baselines for:', list(clear_scores.keys()))

In [ ]:
from ultralytics import YOLO, RTDETR

yolov11 = YOLO('../checkpoints/yolov11/run/weights/best.pt')
yolov12 = YOLO('../checkpoints/yolov12/run/weights/best.pt')
rtdetr  = RTDETR('../checkpoints/rtdetr/run/weights/best.pt')

# TODO: Load RF-DETR checkpoint
# from rfdetr import RFDETRMedium
# rfdetr_model = RFDETRMedium()
# rfdetr_model.load('../checkpoints/rfdetr/best.pt')

In [ ]:
ADVERSE_CONDITIONS = [
    'rainy_day', 'snowy_day', 'night_clear',
    'overcast_day', 'partly_cloudy_day', 'dawn_dusk_clear',
]

for condition in ADVERSE_CONDITIONS:
    RESULTS_DIR = Path(f'../results/{condition}')
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    scores = {}
    robustness = {}

    for name, model in [('yolov11', yolov11), ('yolov12', yolov12), ('rtdetr', rtdetr)]:
        logger.info(f'Evaluating {name} on {condition}...')

        results = model.val(
            data='../configs/dataset.yaml',
            split=f'test_{condition}',
            device=DEVICE,
        )
        m = {**compute_map(results), **compute_precision_recall(results)}
        scores[name] = m

        rob = compute_robustness_metrics(
            clear_map=clear_scores[name]['map50'],
            adverse_map=m['map50'],
        )
        robustness[name] = rob
        logger.info(f'{name} {condition}: mAP50={m["map50"]:.4f}  drop={rob["map_drop"]:.4f}  retention={rob["retention_pct"]:.1f}%')

    # TODO: Add RF-DETR evaluation here once API is confirmed

    (RESULTS_DIR / 'scores.json').write_text(json.dumps(scores, indent=2))
    (RESULTS_DIR / 'robustness.json').write_text(json.dumps(robustness, indent=2))
    print(f'Saved {condition} results to {RESULTS_DIR}')